# Structured credit (tranche + correlation intuition)

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_across_asset_classes.ipynb`. Full **`structured_credit`** waterfall JSON evolves quickly; here we price a **`cds_tranche`** as a **correlation-sensitive tranche** proxy and discuss CLO/ABS concepts in narrative form.


## Concept

- **ABS/CLO/CMBS:** pool cashflows, tranches (attachment/detachment), waterfall priorities.
- **OC/IC tests:** divert cash when collateral quality or interest coverage weakens.

A **CDS tranche** on an index uses **base correlation** to price mezzanine/equity risk—useful for the same dependency modeling mindset.


In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF, print_metrics, usd_ois_curve
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import (
    BaseCorrelationCurve,
    CreditIndexData,
    HazardCurve,
    MarketContext,
)

print("Imports OK (structured credit / tranche).")


Imports OK (structured credit / tranche).


## Instrument JSON (`cds_tranche` proxy)

**Attachment / detachment:** In this JSON, `attach_pct` and `detach_pct` are **index notional fractions in percentage points** on the same scale as base-correlation `detachment_points` (e.g. `detach_pct: 3.0` means a **3%** detachment / 0–3% tranche; in decimal terms that width is **0.03**).

**Equity tranche coupon:** Under **ISDA** standardized tranche conventions, **equity** (first-loss) tranches often carry a **higher running coupon** than mezzanine slices; the demo below uses **500 bp** as a stylized equity coupon for illustration.


In [2]:
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

tranche = json.dumps(
    {
        "type": "cds_tranche",
        "spec": {
            "id": "CDX-0-3",
            "index_name": "CDX.NA.IG",
            "series": 42,
            "attach_pct": 0.0,
            "detach_pct": 3.0,
            "notional": {"amount": "10000000", "currency": "USD"},
            "maturity": "2029-12-20",
            "running_coupon_bp": 500.0,
            "frequency": {"count": 3, "unit": "months"},
            "day_count": "Act360",
            "bdc": "following",
            "calendar_id": "weekends_only",
            "discount_curve_id": "USD-OIS",
            "credit_index_id": "CDX.NA.IG.HAZARD",
            "side": "buy_protection",
            "accumulated_loss": 0.0,
            "standard_imm_dates": False,
            "attributes": {"tags": [], "meta": {}},
        },
    }
)

validate_instrument_json(tranche)
print("cds_tranche JSON OK")


cds_tranche JSON OK


## MarketContext (hazard + base correlation + credit index)


In [3]:
ois = usd_ois_curve(AS_OF)
haz = HazardCurve(
    "CDX-HAZ",
    AS_OF,
    [(1.0, 0.01), (5.0, 0.02), (10.0, 0.025)],
    recovery_rate=0.4,
)
base_correlation = BaseCorrelationCurve(
    "CDX-BC",
    [(3.0, 0.18), (7.0, 0.28), (10.0, 0.35), (15.0, 0.45), (30.0, 0.65), (100.0, 0.85)],
)
credit_index = CreditIndexData(125, 0.4, haz, base_correlation)
mc = MarketContext().insert(ois).insert(haz).insert(base_correlation)
mc.insert_credit_index("CDX.NA.IG.HAZARD", credit_index)
market_json = mc.to_json()
print("credit_indices:", len(json.loads(market_json)["credit_indices"]))

credit_indices: 1


## Pricing


In [4]:
out = price_instrument_with_metrics(tranche, market_json, AS_OF_STR, model="hazard_rate")
print(ValuationResult.from_json(out))


ValuationResult(id="CDX-0-3", price=6179682.3038, currency=USD, metrics=0)


## Metrics


In [5]:
metrics = ["correlation01", "expected_loss", "jump_to_default"]
out = price_instrument_with_metrics(
    tranche,
    market_json,
    AS_OF_STR,
    model="hazard_rate",
    metrics=metrics,
)
print_metrics(ValuationResult.from_json(out), metrics)
print("CLO waterfall JSON: see finstack-quant/valuations/schemas/instruments/1/fixed_income/structured_credit.schema.json")


  correlation01: -104866.16479808
  expected_loss: 7937442.458244368
  jump_to_default: 1600000.0
CLO waterfall JSON: see finstack-quant/valuations/schemas/instruments/1/fixed_income/structured_credit.schema.json


## Takeaways

- **Tranche PV** depends on **hazard** and **base correlation** shapes.
- **Structured credit deals** add pool-specific waterfalls—validate against the latest schema before relying on production JSON.
- Use this **cds_tranche** exercise as a **correlation / subordination** teaching analog.
